# Solutions · 00 · Python for separation processes

Worked solutions to the exercises of [notebook 00](../notebooks/00_python_for_separations.ipynb). Try each exercise
yourself before reading its solution - the learning happens in the attempt. Solutions are one way to
answer each question; other correct approaches exist.

In [1]:
try:                      # on Google Colab (or anywhere engsep is missing): install it from GitHub
    import engsep
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ktwyw/engsep"], check=True)
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import optimize
from engsep import vle
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 90, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

import inspect
from IPython.display import Code

def show_source(obj):
    """Display the source code of a library function or class."""
    return Code(inspect.getsource(obj), language="python")

def heading(text):
    print(f"\n{text}\n" + "=" * len(text))

## Exercise 1

Compute the boiling points of every compound in the Antoine table and rank them. Which pairs of adjacent boilers would be the hardest to separate by distillation, and why does a boiling-point difference not tell the whole story?

In [2]:
bp = pd.Series({n: vle.boiling_point(n) - 273.15 for n in vle.ANTOINE}).sort_values()
print(bp.round(1).to_string())
gaps = bp.diff().dropna()
print("\nclosest adjacent boilers:", ", ".join(f"{gaps.index[i]} / {bp.index[list(bp.index).index(gaps.index[i]) - 1]} ({g:.1f} K)" for i, g in enumerate(gaps) if g < 2.5))

propane                 -42.1
ammonia                 -33.4
isobutane               -11.7
n-butane                 -0.5
diethyl ether            34.6
n-pentane                36.0
acetone                  56.1
methyl acetate           56.9
chloroform               61.2
methanol                 64.5
n-hexane                 68.7
carbon tetrachloride     76.5
ethyl acetate            77.1
ethanol                  78.3
2-butanone               79.6
benzene                  80.1
cyclohexane              80.7
2-propanol               82.3
1-propanol               97.2
n-heptane                98.4
water                   100.0
1,4-dioxane             101.3
toluene                 110.6
1-butanol               117.7
acetic acid             117.9
n-octane                125.7
ethylbenzene            136.2
p-xylene                138.4
o-xylene                144.4
styrene                 145.6
n-decane                174.2

closest adjacent boilers: n-pentane / diethyl ether (1.4 K), methyl a

Several pairs boil within two kelvin (the xylenes, benzene and cyclohexane), which makes distillation hard
but not impossible: what matters is the relative volatility, the ratio of vapour pressures, which can differ
from unity even at equal boiling points if the vapour-pressure curves have different slopes - and non-ideality
(benzene-cyclohexane forms an azeotrope) can help or kill it. Boiling points are the first look, not the
answer.

## Exercise 2

A feed of 200 kmol/h with 0.6 benzene is split into a distillate of 0.98 and a bottoms of 0.02. Compute D, B and the recovery of each component in its product, and repeat for a distillate of 0.999 - how much more benzene is lost to the bottoms when the top is made purer at fixed D?

In [3]:
F, zF = 200.0, 0.6
for xD in (0.98, 0.999):
    xB = 0.02
    D = F * (zF - xB) / (xD - xB); B = F - D
    print(f"xD = {xD}: D = {D:.1f}, B = {B:.1f} kmol/h; benzene recovery in D {D*xD/(F*zF):.4f}; toluene recovery in B {B*(1-xB)/(F*(1-zF)):.4f}; benzene lost to bottoms {B*xB:.2f} kmol/h")

xD = 0.98: D = 120.8, B = 79.2 kmol/h; benzene recovery in D 0.9868; toluene recovery in B 0.9698; benzene lost to bottoms 1.58 kmol/h
xD = 0.999: D = 118.5, B = 81.5 kmol/h; benzene recovery in D 0.9864; toluene recovery in B 0.9985; benzene lost to bottoms 1.63 kmol/h


Making the distillate purer at a fixed bottoms composition shrinks the distillate slightly and leaves the
same benzene in the bottoms: the loss is set by $x_B$, not by $x_D$. The two purities are independent
specifications with independent costs, and a plant pays for whichever the market wants.

## Exercise 3

**Implement it yourself:** write a bubble-point function for any binary with Raoult's law using brentq and compare with `vle.bubble_T` for three compositions of benzene-toluene.

In [4]:
def bubble_T(pair, x1, P=101325.0):
    c1, c2 = pair
    return optimize.brentq(lambda T: x1 * vle.antoine_pressure(c1, T) + (1 - x1) * vle.antoine_pressure(c2, T) - P, 250, 500)
for x1 in (0.2, 0.5, 0.8):
    print(f"x = {x1}: by hand {bubble_T(('benzene', 'toluene'), x1) - 273.15:.3f} degC, library {vle.bubble_T(('benzene', 'toluene'), x1, ideal=True) - 273.15:.3f}")

x = 0.2: by hand 102.103 degC, library 102.103
x = 0.5: by hand 92.112 degC, library 92.112
x = 0.8: by hand 84.380 degC, library 84.380


Raoult's law, two Antoine curves and a root finder: the whole bubble-point calculation in three lines,
identical to the library's.